# 응용 06 · confidence별 검출과 TP·FP·FN 기록

셀은 위에서 아래로 **Shift+Enter**로 실행합니다. 계산·표시에는 pandas가 필요합니다. 이 문서는 제공된 square·circle·triangle 도형 데이터만 다룹니다.

결과 파일 자체는 Git에 포함하지 않습니다. 대신 문서 작성 시 읽은 관측 수치를 코드 셀에 기록해, outputs가 없는 환경에서도 표와 계산을 볼 수 있게 했습니다. 저장된 수치는 과거 실행의 기록이며 문서를 다시 실행해 모델을 학습하는 것은 아닙니다. 실제 모델 호출은 Markdown 코드 예제로 제공합니다. 검증 여부·제출용 판단은 직접 작성하세요.

## 문서의 범위와 근거

이 폴더의 `06_predict.py`를 따라 입력 고정 → confidence별 추론 → 좌표 복원 → 정답 매칭 → 수치 집계 순서로 설명합니다. 입력은 `data/images/val/`과 `data/labels/val/`, 사용 가중치는 `outputs/cli_smoke/weights/last.pt`, 관측 수치는 `outputs/confidence_comparison.csv`에 근거합니다.

## 1. 이번 비교에서 고정한 것과 바꾼 것

|항목|값|
|---|---|
|모델|04번의 cli_smoke/weights/last.pt|
|입력|동일한 val 이미지 12장|
|정답|같은 이름의 val 라벨, 객체 30개|
|입력 크기|imgsz=160|
|문서 작성 시 실행 장치|CPU|
|정답 매칭 IoU|0.50|
|바꾼 값|confidence 0.15, 0.25, 0.50|

confidence는 예측 점수를 걸러내는 기준이고 IoU는 박스의 겹침 비율입니다. 입력 필터 기준과 정답 매칭 기준은 서로 다른 단계입니다. 이번 6번은 5번과 같은 last.pt를 사용해 학습 결과를 이어 봅니다.

In [1]:
from pathlib import Path
import pandas as pd

roots = [Path.cwd(), *Path.cwd().parents]
BASE = next((root / "20261106/02_advanced" for root in roots
             if (root / "20261106/02_advanced/04_train.py").is_file()), None)
if BASE is None:
    raise FileNotFoundError("저장소 안에서 문서를 실행하세요.")
print("실습 폴더:", BASE)


실습 폴더: /home/swchoi/projects/python/practice_assignment/20261106/02_advanced


## 2. predict와 stream=True

A06은 다음과 같이 `results` 변수를 만듭니다.

```python
results = model.predict(
    source=str(BASE / "data/images/val"), conf=conf, stream=True,
    imgsz=160, device=device, save=False, verbose=False
)
```

`source`는 입력 위치이고 `conf`는 현재 반복의 기준입니다. `stream=True`면 결과를 한꺼번에 리스트로 만들지 않고 generator로 순서대로 꺼냅니다. `for result in results:`로 결과를 소비하므로 같은 generator를 두 번 순회해 결과를 다시 얻을 수 없습니다. 각 confidence마다 predict를 새로 호출합니다.

`save=False`는 예측 이미지를 따로 저장하지 않는 설정입니다. 요약 CSV 저장은 그와 별도로 Python 코드에서 수행합니다. 파일을 import할 때 추론이 시작되지 않도록 `main()`과 실행 가드를 사용합니다.

## 3. Results와 Tensor를 읽기

|속성|뜻|
|---|---|
|result.path|현재 이미지의 경로|
|result.orig_shape|원본 높이·가로 (height,width)|
|result.boxes.xyxy|박스 좌표 Tensor|
|result.boxes.cls|예측 클래스 번호 Tensor|
|result.boxes.conf|예측 점수 Tensor|

`.cpu()`는 Tensor를 CPU로 옮깁니다. `.tolist()`는 Tensor를 Python 리스트로 바꿉니다. `torch.argsort(..., descending=True, stable=True)`는 점수가 높은 순으로 위치 번호를 정렬하고 동점은 기존 순서를 유지합니다.

검출이 0개여도 이미지 처리 횟수 `images`를 먼저 증가시킵니다. 객체 행만 세면 처리하지 않은 이미지와 검출 0개 이미지를 구분할 수 없기 때문입니다.

## 4. 정규화 정답을 원본 박스로 되돌리기

YOLO 라벨의 중심(cx,cy)과 크기(bw,bh)는 정규화 값입니다. x축에는 원본 가로, y축에는 원본 세로를 곱합니다.

- x1=(cx−bw/2)×width
- y1=(cy−bh/2)×height
- x2=(cx+bw/2)×width
- y2=(cy+bh/2)×height

아래 계산은 모델을 실행하지 않고 정답 파일의 첫 행을 복원합니다.

In [2]:
label_path = next(p for p in sorted((BASE / "data/labels/val").glob("*.txt")) if p.read_text().strip())
line = label_path.read_text().splitlines()[0]
cls, cx, cy, bw, bh = map(float, line.split())
width, height = 160, 128  # 제공된 원본 도형 이미지 크기
xyxy = [(cx - bw / 2) * width, (cy - bh / 2) * height,
        (cx + bw / 2) * width, (cy + bh / 2) * height]
print("라벨 파일:", label_path.name)
print("정규화 라벨:", line)
print("복원한 xyxy:", xyxy)


라벨 파일: standard_val_000.txt
정규화 라벨: 0 0.18750000 0.34375000 0.20000000 0.25000000
복원한 xyxy: [14.0, 28.0, 46.0, 60.0]


## 5. 점수 순으로 정답을 한 번씩 연결하기

매칭은 이미지 하나 안에서 수행합니다. 같은 클래스이고 아직 사용되지 않은 정답만 후보로 고릅니다. `box_iou()`가 계산한 IoU가 큰 후보부터 보고 0.50 이상이면 하나를 연결합니다.

`used`는 이미 연결한 정답 번호를 담은 집합입니다. `matched=len(used)`가 TP 수가 됩니다. FP는 예측 수−TP, FN은 정답 수−TP입니다. 정답 하나에 중복 예측을 여러 개 TP로 연결하지 않습니다. 정답이 없는 이미지의 예측은 FP에 들어갑니다. 예측이 없는 이미지의 정답은 FN에 들어갑니다.

`reshape(-1,4)`는 빈 정답도 0×4 형태의 좌표 배열로 표현합니다. 분모가 0이면 코드에서 precision·recall을 0으로 기록합니다. 이는 정의되지 않은 0/0을 처리하기 위해 정한 기록 규칙입니다.

## 6. 이번 실행의 관측값

관측값 출처는 `outputs/confidence_comparison.csv`이며, 문서 작성 중 6번 코드를 실행해 생성했습니다. 입력 12장에는 객체 30개가 있습니다.

In [3]:
confidence_records = [{'conf': 0.15, 'images': 12.0, 'detections': 0.0, 'tp': 0.0, 'fp': 0.0, 'fn': 30.0, 'precision': 0.0, 'recall': 0.0}, {'conf': 0.25, 'images': 12.0, 'detections': 0.0, 'tp': 0.0, 'fp': 0.0, 'fn': 30.0, 'precision': 0.0, 'recall': 0.0}, {'conf': 0.5, 'images': 12.0, 'detections': 0.0, 'tp': 0.0, 'fp': 0.0, 'fn': 30.0, 'precision': 0.0, 'recall': 0.0}]
comparison = pd.DataFrame(confidence_records)
for col in ["images", "detections", "tp", "fp", "fn"]:
    comparison[col] = comparison[col].astype(int)
print(comparison.to_string(index=False))


 conf  images  detections  tp  fp  fn  precision  recall
 0.15      12           0   0   0  30        0.0     0.0
 0.25      12           0   0   0  30        0.0     0.0
 0.50      12           0   0   0  30        0.0     0.0


세 confidence 모두 처리 이미지 12장, 검출 0개, TP 0, FP 0, FN 30입니다. 따라서 recall=0/30=0이며 precision은 0/0 대신 코드의 규칙에 따라 0으로 기록됩니다.

**검출이 없어서 FP=0인 상태**를 precision이 높은 모델이라는 뜻으로 해석하지 않습니다. 이번 세 기준에서 검출이 없었다는 관측이며, 더 낮은 기준에서도 예측이 없다고 일반화하지 않습니다. confidence를 바꿨는데 수치가 같다는 사실만으로 세 기준이 모든 모델에서 동등하다고 볼 수도 없습니다.

In [4]:
calculated = comparison.copy()
calculated["정답 객체 수(TP+FN)"] = calculated["tp"] + calculated["fn"]
calculated["검출 수(TP+FP)"] = calculated["tp"] + calculated["fp"]
calculated["recall 계산값"] = calculated.apply(
    lambda row: row["tp"] / (row["tp"] + row["fn"]) if row["tp"] + row["fn"] else 0, axis=1)
print(calculated[["conf", "images", "정답 객체 수(TP+FN)", "검출 수(TP+FP)", "recall 계산값"]].to_string(index=False))


 conf  images  정답 객체 수(TP+FN)  검출 수(TP+FP)  recall 계산값
 0.15      12              30            0         0.0
 0.25      12              30            0         0.0
 0.50      12              30            0         0.0


## 7. 05번 결과와 연결해서 읽기

05번은 평가기의 여러 confidence 순위와 IoU 기준에서 AP를 계산합니다. 06번은 지정한 confidence 한 개와 정답 매칭 IoU 0.50에서 TP·FP·FN을 셉니다. 이번에는 05번의 mAP가 0이고 06번의 세 기준에서 검출도 0이라는 각각의 기록을 얻었습니다. 두 절차가 같은 지표를 계산하는 것은 아닙니다.

`outputs` 없이 이 문서만 실행하면 저장해 둔 표와 계산을 다시 볼 수 있습니다. 실제 모델을 다시 실행하려면 먼저 04번 학습으로 체크포인트를 준비한 후 05번과 06번을 실행하세요. 기존 결과를 재사용할 때는 학습 설정과 체크포인트 경로도 함께 기록하세요.

## 8. 직접 기록

|항목|직접 작성|
|---|---|
|실행·검증 여부와 근거||
|관측한 수치||
|추가 확인이 필요한 원인 가설||
|다음 비교의 고정 조건과 변경 조건||

연습: TP=2, FP=1, FN=1인 경우 precision과 recall을 각각 계산해 보세요. 다음에는 TP=0, FP=0, FN=30인 이번 기록과 분모가 어떻게 다른지 비교해 보세요.

참고한 로컬 파일: `06_predict.py`, `data/images/val/`, `data/labels/val/`, `outputs/confidence_comparison.csv`.